<a href="https://www.kaggle.com/code/mendelowitzadi/isic2018-task1-segmentation?scriptVersionId=316521217" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ISIC 2018 Task 1: Lesion Segmentation with Mask R-CNN

Instance segmentation of skin lesions using Mask R-CNN (ResNet-50-FPN backbone).

**Dataset:** ISIC 2018 Challenge Task 1 — Lesion Boundary Segmentation  
**Reference:** Codella et al., arXiv:1902.03368, 2019  
**Kaggle dataset:** `tschandl/isic2018-challenge-task1-data-segmentation`

**Model:** `torchvision.models.detection.maskrcnn_resnet50_fpn` pretrained on COCO,  
with box and mask predictor heads replaced for 2 classes (background + lesion).

**Primary metric:** Thresholded Jaccard index (T = 0.65), per Codella et al. (2019).  
Per-image IoU values below T are set to zero before averaging, penalising gross failures.

## 1. Imports and configuration

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.utils.data
import torchvision.transforms.v2.functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
BASE = Path("/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation")

CFG = {
    # Paths
    "img_dir": BASE / "ISIC2018_Task1-2_Training_Input",
    "mask_dir": BASE / "ISIC2018_Task1_Training_GroundTruth",
    "ckpt_dir": Path("/kaggle/working/checkpoints"),
    # Data
    "val_fraction": 0.2,  # fraction of training data held out for validation
    "img_size": 512,  # images resized to (img_size x img_size)
    "seed": 42,
    # Training
    "num_classes": 2,  # background (0) + lesion (1)
    "batch_size": 4,
    "num_epochs": 20,
    "lr": 5e-4,
    "weight_decay": 1e-4,
    "lr_step_size": 7,  # StepLR: decay every N epochs
    "lr_gamma": 0.5,
    # Evaluation
    "mask_threshold": 0.5,  # sigmoid threshold for binary mask prediction
    "jaccard_threshold": 0.65,  # Thresholded Jaccard T, per Codella et al. (2019)
    "score_threshold": 0.5,  # minimum detection score to consider
}

CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Images : {CFG['img_dir']}")
print(f"Masks  : {CFG['mask_dir']}")

## 2. Dataset

In [ ]:
class ISICSegmentationDataset(torch.utils.data.Dataset):
    """
    ISIC 2018 Task 1 lesion segmentation dataset.

    Each sample yields the Mask R-CNN target format:
        image : FloatTensor (3, H, W), values in [0, 1]
        target: dict with keys
            boxes  -- FloatTensor (1, 4), x1 y1 x2 y2 in pixel coords
            labels -- LongTensor  (1,),   always 1 (lesion class)
            masks  -- BoolTensor  (1, H, W)

    Bounding boxes are derived from the tight bounding rectangle of the binary mask.
    Images with no foreground pixels are skipped at construction time.
    Note: _filter_empty_masks opens all mask files at init; this is a one-time cost of ~2,594 file reads before training begins.

    Args:
        img_dir:  Directory containing ISIC_XXXXXXX.jpg files.
        mask_dir: Directory containing ISIC_XXXXXXX_segmentation.png files.
        img_ids:  List of ISIC IDs (e.g. 'ISIC_0024306') to include.
        img_size: Both spatial dimensions are resized to this value.
        augment:  If True, apply random horizontal flip augmentation.
    """

    def __init__(
        self, img_dir: Path, mask_dir: Path, img_ids: list[str], img_size: int = 512, augment: bool = False
    ) -> None:
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.augment = augment

        # Keep only IDs whose mask contains at least one foreground pixel.
        self.img_ids = self._filter_empty_masks(img_ids)
        print(
            f"Dataset: {len(self.img_ids)} valid samples " f"(removed {len(img_ids) - len(self.img_ids)} empty masks)"
        )

    def _filter_empty_masks(self, img_ids: list[str]) -> list[str]:
        valid = []
        for img_id in img_ids:
            mask_path = self.mask_dir / f"{img_id}_segmentation.png"
            mask = np.array(Image.open(mask_path).convert("L"))
            if mask.max() > 0:
                valid.append(img_id)
        return valid

    def __len__(self) -> int:
        return len(self.img_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, dict]:
        img_id = self.img_ids[idx]

        # Load image as float tensor in [0, 1].
        img = Image.open(self.img_dir / f"{img_id}.jpg").convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.Resampling.BILINEAR)
        img_tensor = TF.to_image(img)
        img_tensor = TF.to_dtype(img_tensor, dtype=torch.float32, scale=True)

        # Load mask as binary array, then derive bounding box from its extent.
        mask = Image.open(self.mask_dir / f"{img_id}_segmentation.png").convert("L")
        mask = mask.resize((self.img_size, self.img_size), Image.Resampling.NEAREST)
        mask_np = (np.array(mask) > 127).astype(np.uint8)

        rows = np.any(mask_np, axis=1)
        cols = np.any(mask_np, axis=0)
        y1, y2 = np.where(rows)[0][[0, -1]]
        x1, x2 = np.where(cols)[0][[0, -1]]
        boxes = torch.tensor([[x1, y1, x2, y2]], dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": torch.ones(1, dtype=torch.long),
            "masks": torch.from_numpy(mask_np).unsqueeze(0).bool(),
        }

        # Augmentation: random horizontal flip applied consistently to image, mask, and bounding box.
        if self.augment and random.random() > 0.5:
            img_tensor = TF.horizontal_flip(img_tensor)
            target["masks"] = TF.horizontal_flip(target["masks"])
            W = self.img_size
            x1f = W - boxes[0, 2]
            x2f = W - boxes[0, 0]
            target["boxes"] = torch.tensor([[x1f, boxes[0, 1], x2f, boxes[0, 3]]], dtype=torch.float32)

        return img_tensor, target

In [ ]:
def collate_fn(batch: list[tuple[torch.Tensor, dict]]) -> tuple[list[torch.Tensor], list[dict]]:
    """Collate function for Mask R-CNN: returns lists, not stacked tensors."""
    images, targets = zip(*batch)
    return list(images), list(targets)


def build_dataloaders(cfg: dict) -> tuple[torch.utils.data.DataLoader, torch.utils.data.DataLoader]:
    """Split training images 80/20 and return train and validation DataLoaders."""
    all_ids = sorted(p.stem for p in cfg["img_dir"].glob("*.jpg"))
    rng = random.Random(cfg["seed"])
    rng.shuffle(all_ids)

    split = int(len(all_ids) * (1 - cfg["val_fraction"]))
    train_ids, val_ids = all_ids[:split], all_ids[split:]

    train_ds = ISICSegmentationDataset(
        cfg["img_dir"], cfg["mask_dir"], train_ids, img_size=cfg["img_size"], augment=True
    )
    val_ds = ISICSegmentationDataset(cfg["img_dir"], cfg["mask_dir"], val_ids, img_size=cfg["img_size"], augment=False)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=2, collate_fn=collate_fn, pin_memory=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True
    )
    print(f"Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")
    return train_loader, val_loader


train_loader, val_loader = build_dataloaders(CFG)

## 3. Dataset sanity check

In [ ]:
def visualise_sample(loader: torch.utils.data.DataLoader, n: int = 3) -> None:
    """Plot n images with their ground-truth mask and bounding box overlaid."""
    images, targets = next(iter(loader))
    fig, axes = plt.subplots(1, min(n, len(images)), figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, img, tgt in zip(axes, images[:n], targets[:n]):
        # img: (3, H, W) float [0,1]
        ax.imshow(img.permute(1, 2, 0).numpy())
        mask = tgt["masks"][0].numpy()  # (H, W)
        ax.imshow(mask, alpha=0.35, cmap="Reds")
        x1, y1, x2, y2 = tgt["boxes"][0].tolist()
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


visualise_sample(train_loader, n=3)

## 4. Model

In [ ]:
def build_model(num_classes: int) -> torch.nn.Module:
    """
    Load Mask R-CNN (ResNet-50-FPN) pretrained on COCO and replace the box and mask predictor heads for `num_classes` output classes.

    Both heads are replaced rather than fine-tuned because the output dimension changes (91 COCO classes -> 2 classes).
    The backbone and FPN weights are retained.

    Args:
        num_classes: Number of output classes including background.

    Returns:
        Mask R-CNN model ready for fine-tuning.
    """
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)

    # Replace box predictor head.
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)

    # Replace mask predictor head; preserve the hidden dim from the existing head.
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_dim = model.roi_heads.mask_predictor.conv5_mask.out_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_dim, num_classes)

    return model


model = build_model(CFG["num_classes"]).to(DEVICE)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

## 5. Evaluation metric: Thresholded Jaccard

In [ ]:
def compute_jaccard(pred_mask: np.ndarray, gt_mask: np.ndarray, threshold: float = 0.65) -> tuple[float, float]:
    """
    Compute raw IoU and thresholded Jaccard for a single image pair.

    The thresholded Jaccard sets IoU to zero if it falls below T=0.65, per the ISIC 2018 evaluation protocol (Codella et al., 2019). T was derived from
    inter-observer variability on the 2016 challenge data (minimum pairwise agreement: 0.743).

    Every ISIC 2018 image contains at least one lesion pixel; union == 0 indicates a corrupt sample and is penalised with (0.0, 0.0) rather than
    rewarded with a perfect score.

    Args:
        pred_mask: Binary predicted mask, shape (H, W), dtype bool or uint8.
        gt_mask: Binary ground-truth mask, shape (H, W), dtype bool or uint8.
        threshold: Jaccard threshold below which score is set to zero.

    Returns:
        Tuple of (raw_iou, thresholded_jaccard).
    """
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    intersection = (pred & gt).sum()
    union = (pred | gt).sum()

    if union == 0:
        return 0.0, 0.0

    iou = float(intersection / union)
    jaccard = iou if iou >= threshold else 0.0
    return iou, jaccard


@torch.no_grad()
def evaluate(model: torch.nn.Module, loader: torch.utils.data.DataLoader, cfg: dict) -> float:
    """
    Evaluate the model on  and return mean Thresholded Jaccard.

    For each image the highest-scoring detection above score_threshold is selected as the predicted lesion.
    If no detection passes the threshold, the predicted mask is all zeros.

    Args:
        model: Mask R-CNN model (set to eval mode internally).
        loader: DataLoader with batch_size=1.
        cfg: Configuration dict.

    Returns:
        Mean Thresholded Jaccard across all validation images.
    """
    model.eval()
    scores = []

    for images, targets in loader:
        images = [img.to(DEVICE) for img in images]
        outputs = model(images)

        for output, target in zip(outputs, targets):
            gt_mask = target["masks"][0].numpy()
            H, W = gt_mask.shape

            pred_mask = np.zeros((H, W), dtype=bool)
            keep = output["scores"] >= cfg["score_threshold"]
            if keep.any():
                best_idx = output["scores"][keep].argmax()
                pred_mask = output["masks"][keep][best_idx, 0].cpu().numpy() >= cfg["mask_threshold"]

            _, jaccard = compute_jaccard(pred_mask, gt_mask, cfg["jaccard_threshold"])
            scores.append(jaccard)

    return float(np.mean(scores))

## 6. Training

In [ ]:
def train_one_epoch(
    model: torch.nn.Module, loader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer
) -> dict[str, float]:
    """
    Run one training epoch.

    Mask R-CNN in train mode returns a loss dict with keys:
        loss_classifier, loss_box_reg, loss_mask, loss_objectness, loss_rpn_box_reg.
    The total loss is their sum.

    Returns:
        Dict of mean loss values for the epoch.
    """
    model.train()
    epoch_losses: dict[str, list[float]] = {}

    for images, targets in loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        total_loss = sum(loss_dict.values())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        for k, v in loss_dict.items():
            epoch_losses.setdefault(k, []).append(v.item())
        epoch_losses.setdefault("total", []).append(total_loss.item())

    return {k: float(np.mean(v)) for k, v in epoch_losses.items()}


def train(
    model: torch.nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    cfg: dict,
) -> tuple[dict, int]:
    """
    Full training loop with LR scheduling and best-checkpoint saving.

    If cfg['resume_ckpt'] points to an existing file, resumes from that checkpoint and runs cfg['num_epochs'] additional epochs.
    Otherwise trains from scratch for cfg['num_epochs'] epochs.

    Args:
        model: Mask R-CNN model.
        train_loader: Training DataLoader.
        val_loader: Validation DataLoader (batch_size=1).
        cfg: Configuration dict.

    Returns:
        Tuple of (history dict, start_epoch).
    """
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=cfg["lr_step_size"], gamma=cfg["lr_gamma"])

    history: dict[str, list[float]] = {"train_loss": [], "val_jaccard": []}
    best_jaccard, start_epoch = 0.0, 1

    if cfg.get("resume_ckpt") and Path(cfg["resume_ckpt"]).exists():
        ckpt = torch.load(cfg["resume_ckpt"], map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_jaccard = ckpt["val_jaccard"]
        start_epoch = ckpt["epoch"] + 1
        for _ in range(ckpt["epoch"]):
            scheduler.step()
        print(f"Resumed from epoch {ckpt['epoch']} (val_jaccard={best_jaccard:.4f})")

    final_epoch = start_epoch + cfg["num_epochs"] - 1

    for epoch in range(start_epoch, start_epoch + cfg["num_epochs"]):
        losses = train_one_epoch(model, train_loader, optimizer)
        scheduler.step()
        val_jaccard = evaluate(model, val_loader, cfg)

        history["train_loss"].append(losses["total"])
        history["val_jaccard"].append(val_jaccard)

        print(
            f"Epoch {epoch:02d}/{final_epoch} | "
            f"loss_total={losses['total']:.4f} | "
            f"loss_mask={losses.get('loss_mask', 0):.4f} | "
            f"loss_box={losses.get('loss_box_reg', 0):.4f} | "
            f"loss_cls={losses.get('loss_classifier', 0):.4f} | "
            f"val_jaccard={val_jaccard:.4f}"
        )

        if val_jaccard > best_jaccard:
            best_jaccard = val_jaccard
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_jaccard": best_jaccard,
                },
                cfg["ckpt_dir"] / "best_maskrcnn_isic2018.pth",
            )
            print(f"  Checkpoint saved (val_jaccard={best_jaccard:.4f})")

    print(f"\nTraining complete. Best val Thresholded Jaccard: {best_jaccard:.4f}")
    return history, start_epoch


CFG["resume_ckpt"] = None

history, start_epoch = train(model, train_loader, val_loader, CFG)

## 7. Training curves

In [ ]:
epochs = range(start_epoch, start_epoch + len(history["train_loss"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], marker="o", label="Train total loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history["val_jaccard"], marker="o", color="darkorange", label="Val Thresholded Jaccard")
ax2.axhline(0.65, linestyle="--", color="grey", alpha=0.6, label="T = 0.65")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Thresholded Jaccard")
ax2.set_title("Validation Thresholded Jaccard")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(Path("/kaggle/working") / "training_curves.png", dpi=150)
plt.show()

## 8. Qualitative evaluation on validation set

In [ ]:
# Load best checkpoint
ckpt = torch.load(CFG["ckpt_dir"] / "best_maskrcnn_isic2018.pth", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_jaccard={ckpt['val_jaccard']:.4f})")

## 9. Final validation score

In [ ]:
final_jaccard = evaluate(model, val_loader, CFG)
print(f"Final val Thresholded Jaccard (T=0.65): {final_jaccard:.4f}")
print(f"Best val Thresholded Jaccard during training: {ckpt['val_jaccard']:.4f}")

In [ ]:
@torch.no_grad()
def evaluate_segmentation(
    model: torch.nn.Module, val_loader: torch.utils.data.DataLoader, cfg: dict
) -> tuple[float, list[dict]]:
    """
    Evaluate Mask R-CNN on the validation set and collect per-image results.

    The DataLoader uses Mask R-CNN's collate_fn, returning a list of tensors per batch rather than a stacked batch tensor.

    Args:
        model: Mask R-CNN model (set to eval mode internally).
        val_loader: Validation DataLoader yielding (list[Tensor], list[dict]).
        cfg: Configuration dict supplying score_threshold, mask_threshold, and jaccard_threshold.

    Returns:
        (mean_thresholded_jaccard, results) where results is a list of dicts
        with keys: image (H,W,3 float32), gt_mask (H,W uint8),
        pred_mask (H,W uint8), iou (float), jaccard (float).
    """
    model.eval()
    results = []

    for images, targets in val_loader:
        images = [img.to(DEVICE) for img in images]
        outputs = model(images)

        for img, target, output in zip(images, targets, outputs):
            gt_mask = target["masks"][0].cpu().numpy().astype(np.uint8)
            H, W = gt_mask.shape

            # Mask R-CNN returns detections sorted by score descending. Accept the top detection if it meets the confidence threshold.
            pred_mask = np.zeros((H, W), dtype=np.uint8)
            if len(output["scores"]) > 0 and output["scores"][0] >= cfg["score_threshold"]:
                soft_mask = output["masks"][0, 0].cpu().numpy()
                pred_mask = (soft_mask >= cfg["mask_threshold"]).astype(np.uint8)

            iou, jaccard = compute_jaccard(pred_mask, gt_mask, cfg["jaccard_threshold"])

            # Recover display image from float [0,1] tensor (no normalisation was applied - Mask R-CNN takes [0,1] inputs directly).
            img_np = img.cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0)

            results.append(
                {"image": img_np, "gt_mask": gt_mask, "pred_mask": pred_mask, "iou": iou, "jaccard": jaccard}
            )

    mean_jaccard = float(np.mean([r["jaccard"] for r in results]))
    return mean_jaccard, results


def visualise_segmentation_examples(results: list[dict], n: int = 6, save_path: Path | None = None) -> None:
    """
    Plot n examples spanning the IoU distribution: input | ground truth | prediction | error map.

    Examples are sampled evenly from worst to best IoU, covering catastrophic failures, borderline cases, and high-confidence successes.

    Error map convention:
        Green = true positive  (correct lesion pixel)
        Red = false positive (predicted lesion, actually background)
        Blue = false negative (missed lesion pixel)
        Black = true negative  (correct background pixel)

    Args:
        results: Per-image result dicts from evaluate_segmentation.
        n: Number of examples to display (default 6).
        save_path: If provided, save the figure here at 150 dpi.
    """
    sorted_results = sorted(results, key=lambda r: r["iou"])
    n = min(n, len(sorted_results))
    indices = np.linspace(0, len(sorted_results) - 1, n, dtype=int)
    selected = [sorted_results[i] for i in indices]

    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Input image", "Ground truth", "Prediction", "Error map"]):
        axes[0, col].set_title(title, fontsize=11, fontweight="bold")

    for row, r in enumerate(selected):
        gt = r["gt_mask"]
        pred = r["pred_mask"]

        error = np.zeros((*gt.shape, 3), dtype=np.float32)
        error[(gt == 1) & (pred == 1)] = [0.0, 0.8, 0.0]  # TP green
        error[(gt == 0) & (pred == 1)] = [0.8, 0.0, 0.0]  # FP red
        error[(gt == 1) & (pred == 0)] = [0.0, 0.0, 0.8]  # FN blue

        axes[row, 0].imshow(r["image"])
        axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
        axes[row, 3].imshow(error)
        axes[row, 0].set_ylabel(
            f"IoU={r['iou']:.3f}  Jaccard={r['jaccard']:.3f}", fontsize=9, rotation=0, labelpad=130, va="center"
        )
        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved to {save_path}")
    plt.show()


# --- Run evaluation ---
mean_jaccard, results = evaluate_segmentation(model, val_loader, CFG)

n_total = len(results)
n_zero = sum(1 for r in results if r["jaccard"] == 0.0)
print(f"Validation images evaluated: {n_total}")
print(f"Mean thresholded Jaccard (T=0.65): {mean_jaccard:.4f}")
print(f"Images scoring zero (IoU < 0.65): {n_zero} / {n_total} " f"({100 * n_zero / n_total:.1f}%)")

# --- Visualise 6 examples spanning worst to best IoU ---
visualise_segmentation_examples(results, n=6, save_path=Path("/kaggle/working") / "segmentation_examples_task1.png")